# CAP10 WF2020 Return-Preserving Target Revision V1

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` production composer.
- **Reference baseline**: `soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05` for the calibrated wf_2020 floor, with `soxl_backtest_policy_reimplementation_v1` retained as the qlib-aligned buy/sell reference baseline.
- **Direct comparison baseline**: `cap10_three_fold_checkpoint_metric_replay_v2`.
- **Role**: `diagnose` and hand off a target revision; no actor promotion or retention.
- **Target window**: `wf_2020` failed fold, while preserving the existing `wf_2022` and `wf_2023_2024` gates.
- **Accepted trade-off**: use available drawdown headroom to raise 2020 bull participation, but keep MaxDD `<20%` and do not regress YTD/reference alignment.
- **Changed layer**: `selection` / `reward target` / `execution target`, not shared model code.
- **External reference**: qlib-aligned SOXLBacktest buy/sell reference and the calibrated risk-cap floor from `wf2020_reference_floor_gate_calibration_v1`.

## Hypothesis

The latest metric replay is not blocked by drawdown in `wf_2020`; it is blocked by return under-participation. The next useful target should not lower risk further. It should selectively increase 2020 bull/re-entry participation while retaining drawdown drift stops.

## Historical Duplicate Check

- Closest success case: `cap10_three_fold_checkpoint_metric_replay_v2` already passes `wf_2022` and `wf_2023_2024` under MaxDD `<20%`.
- Closest failure case: `wf2020_reference_floor_gate_calibration_v1` shows pure cap-only revisions leave too much return gap and are not enough for YTD/reference alignment.
- Why this is not a duplicate: this card does not rerun another cap-only target or checkpoint selector. It converts the latest failed fold into a return-preserving target revision contract so frontier/dashboard stop pointing at stale diagnostics.


In [1]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise FileNotFoundError("could not find SOXL repo root from notebook cwd")


ROOT = find_repo_root(Path.cwd())
CARD = ROOT / "runtime" / "model_selection" / "card" / "cap10_wf2020_return_preserving_target_revision_v1"
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_SUMMARY_PATH = ROOT / "runtime" / "model_selection" / "card" / "cap10_three_fold_checkpoint_metric_replay_v2" / "summary.json"
CALIBRATION_SUMMARY_PATH = ROOT / "runtime" / "model_selection" / "card" / "wf2020_reference_floor_gate_calibration_v1" / "summary.json"
CALIBRATION_GAP_PATH = ROOT / "runtime" / "model_selection" / "card" / "wf2020_reference_floor_gate_calibration_v1" / "wf2020_reference_floor_gap.csv"
GATE_SUMMARY_PATH = ROOT / "runtime" / "model_selection" / "card" / "model_retention_walkforward_gate_v1" / "summary.json"
BASELINE_METRICS_PATH = ROOT / "runtime" / "model_selection" / "card" / "soxlbacktest_qlib_walkforward_baseline_v1" / "candidate_metrics.csv"
CANDIDATE_METRICS_PATH = ROOT / "runtime" / "model_selection" / "card" / "cap10_three_fold_checkpoint_metric_replay_v2" / "checkpoint_candidate_metrics.csv"

EXPERIMENT = "cap10_wf2020_return_preserving_target_revision_v1"
NEXT_MECHANISM = "build_return_preserving_wf2020_bull_participation_target_then_rerun_three_fold_bc_checkpoint_metric_replay"
DECISION = "wf2020_return_preserving_target_revision_ready_no_promotion"


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    return value

source_summary = load_json(SOURCE_SUMMARY_PATH)
calibration_summary = load_json(CALIBRATION_SUMMARY_PATH)
gate_summary = load_json(GATE_SUMMARY_PATH).get("summary", load_json(GATE_SUMMARY_PATH))
baseline_metrics = pd.read_csv(BASELINE_METRICS_PATH)
candidate_metrics = pd.read_csv(CANDIDATE_METRICS_PATH)
calibration_gap = pd.read_csv(CALIBRATION_GAP_PATH)

selected_by_fold = source_summary["selected_by_fold"]
wf2020 = selected_by_fold["wf_2020"]
source_daily_path = ROOT / wf2020["selected_daily_path"]
daily = pd.read_csv(source_daily_path)
date_col = "datetime" if "datetime" in daily.columns else "date"
daily[date_col] = pd.to_datetime(daily[date_col])
strategy_peak = daily["strategy_nav"].cummax()
daily_max_dd_pct = abs(float((daily["strategy_nav"] / strategy_peak - 1.0).min() * 100.0))
daily_return_pct = float((daily["strategy_nav"].iloc[-1] / daily["strategy_nav"].iloc[0] - 1.0) * 100.0)
daily_buyhold_return_pct = float((daily["buyhold_nav"].iloc[-1] / daily["buyhold_nav"].iloc[0] - 1.0) * 100.0)

source_candidate = candidate_metrics[
    (candidate_metrics["fold"].eq("wf_2020"))
    & (candidate_metrics["candidate"].eq(wf2020["selected_candidate"]))
].iloc[0]
calibrated_gap = calibration_gap.iloc[0]
calibrated_targets = calibration_summary["target_relative_return_by_fold_pp"]

relative_gap_to_calibrated_floor_pp = float(
    calibrated_targets["wf_2020"] - float(wf2020["selected_relative_return"])
)
drawdown_headroom_to_20pct = float(gate_summary["max_walkforward_drawdown_pct"] - float(wf2020["selected_max_dd"]))
return_gap_to_risk_floor_pp = float(
    calibrated_gap["risk_floor_return_pct"] - float(source_candidate["test_return"])
)

risk_cap_ytd = baseline_metrics[
    baseline_metrics["candidate"].eq("risk_cap_35_15_05")
    & baseline_metrics["window"].eq("ytd_2026")
].iloc[0]
legacy_ytd = baseline_metrics[
    baseline_metrics["candidate"].eq("legacy_default")
    & baseline_metrics["window"].eq("ytd_2026")
].iloc[0]

wf2020_gap = pd.DataFrame([
    {
        "source_card": "cap10_three_fold_checkpoint_metric_replay_v2",
        "fold": "wf_2020",
        "selected_candidate": wf2020["selected_candidate"],
        "selected_kind": wf2020["selected_kind"],
        "selected_return_pct": float(source_candidate["test_return"]),
        "gate_reference_return_pct": float(source_candidate["test_buy_and_hold_return"]),
        "selected_relative_return_pp": float(wf2020["selected_relative_return"]),
        "calibrated_relative_target_pp": float(calibrated_targets["wf_2020"]),
        "relative_gap_to_calibrated_floor_pp": relative_gap_to_calibrated_floor_pp,
        "risk_floor_candidate": calibrated_gap["risk_floor_candidate"],
        "risk_floor_return_pct": float(calibrated_gap["risk_floor_return_pct"]),
        "return_gap_to_risk_floor_pp": return_gap_to_risk_floor_pp,
        "selected_max_dd_pct": float(wf2020["selected_max_dd"]),
        "target_max_dd_pct": float(gate_summary["max_walkforward_drawdown_pct"]),
        "drawdown_headroom_to_20pct": drawdown_headroom_to_20pct,
        "daily_path": wf2020["selected_daily_path"],
        "daily_rows": int(len(daily)),
        "daily_start": daily[date_col].min().strftime("%Y-%m-%d"),
        "daily_end": daily[date_col].max().strftime("%Y-%m-%d"),
        "daily_return_pct": daily_return_pct,
        "daily_buyhold_return_pct": daily_buyhold_return_pct,
        "daily_max_dd_pct": daily_max_dd_pct,
        "daily_avg_position": float(daily["position"].mean()),
        "daily_buy_count": int((daily["action"].astype(str).str.upper() == "BUY").sum()),
        "daily_sell_count": int((daily["action"].astype(str).str.upper() == "SELL").sum()),
    }
])

contract = {
    "experiment": EXPERIMENT,
    "decision": DECISION,
    "source_card": "runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2/summary.json",
    "calibration_card": "runtime/model_selection/card/wf2020_reference_floor_gate_calibration_v1/summary.json",
    "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
    "direct_comparison_baseline": "cap10_three_fold_checkpoint_metric_replay_v2",
    "role": "diagnose_target_revision_handoff",
    "changed_layer": ["selection", "reward_target", "execution_target"],
    "target_windows": ["wf_2020", "wf_2022", "wf_2023_2024"],
    "failed_folds": ["wf_2020"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": False,
    "next_training_allowed_after_target_revision": True,
    "primary_failure": "wf_2020_return_underparticipation_after_metric_replay",
    "next_mechanism": NEXT_MECHANISM,
    "acceptance_criteria": {
        "all_required_walkforward_windows_present": gate_summary["required_wf_windows"],
        "max_walkforward_drawdown_pct_lt": gate_summary["max_walkforward_drawdown_pct"],
        "ytd_return_pct_gte_soxlbacktest_reference": gate_summary["reference_return_pct"],
        "ytd_return_pct_gte_current_best_floor": gate_summary["required_best_return_pct"],
        "wf2020_relative_return_pp_gte": calibrated_targets["wf_2020"],
        "wf2022_relative_return_pp_gte": calibrated_targets["wf_2022"],
        "wf2023_2024_relative_return_pp_gte": calibrated_targets["wf_2023_2024"],
    },
    "target_design_constraints": [
        "do_not_repeat_pure_low_risk_cap_only_target",
        "increase_wf2020_bull_participation_or_reentry_before_raising_global_risk",
        "keep_drawdown_drift_stop_and_max_dd_below_20pct",
        "rerun_three_fold_bc_checkpoint_metric_replay_before_any_actor_retention",
    ],
    "evidence": {
        "wf2020_selected_return_pct": float(source_candidate["test_return"]),
        "wf2020_selected_relative_return_pp": float(wf2020["selected_relative_return"]),
        "wf2020_calibrated_relative_target_pp": float(calibrated_targets["wf_2020"]),
        "wf2020_relative_gap_to_calibrated_floor_pp": relative_gap_to_calibrated_floor_pp,
        "wf2020_selected_max_dd_pct": float(wf2020["selected_max_dd"]),
        "wf2020_drawdown_headroom_to_20pct": drawdown_headroom_to_20pct,
        "risk_cap_35_15_05_ytd_return_pct": float(risk_cap_ytd["rulebase_return_pct"]),
        "legacy_soxlbacktest_ytd_return_pct": float(legacy_ytd["rulebase_return_pct"]),
        "required_soxlbacktest_return_pct": gate_summary["reference_return_pct"],
        "required_best_return_pct": gate_summary["required_best_return_pct"],
    },
}

summary = {
    "experiment": EXPERIMENT,
    "decision": DECISION,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": False,
    "next_training_allowed_after_target_revision": True,
    "reference_baseline": contract["reference_baseline"],
    "direct_comparison_baseline": contract["direct_comparison_baseline"],
    "source_card": contract["source_card"],
    "source_daily_path": wf2020["selected_daily_path"],
    "failed_folds": ["wf_2020"],
    "primary_failure": contract["primary_failure"],
    "next_mechanism": NEXT_MECHANISM,
    "target_max_dd_pct": gate_summary["max_walkforward_drawdown_pct"],
    "required_best_return_pct": gate_summary["required_best_return_pct"],
    "required_soxlbacktest_return_pct": gate_summary["reference_return_pct"],
    "target_relative_return_by_fold_pp": calibrated_targets,
    "selected_by_fold": {
        "wf_2020": {
            "selected_candidate": wf2020["selected_candidate"],
            "selected_kind": wf2020["selected_kind"],
            "selected_return_pct": float(source_candidate["test_return"]),
            "selected_relative_return": float(wf2020["selected_relative_return"]),
            "selected_max_dd": float(wf2020["selected_max_dd"]),
            "selected_candidate_gate": bool(wf2020["selected_candidate_gate"]),
            "selected_daily_path": wf2020["selected_daily_path"],
            "calibrated_relative_target_pp": float(calibrated_targets["wf_2020"]),
            "relative_gap_to_calibrated_floor_pp": relative_gap_to_calibrated_floor_pp,
            "drawdown_headroom_to_20pct": drawdown_headroom_to_20pct,
        },
        "wf_2022": selected_by_fold["wf_2022"],
        "wf_2023_2024": selected_by_fold["wf_2023_2024"],
    },
    "wf2020_gap": wf2020_gap.iloc[0].to_dict(),
    "sidecars": [
        "wf2020_return_preserving_gap.csv",
        "target_revision_contract.json",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
}

selection = {
    "experiment": EXPERIMENT,
    "decision": DECISION,
    "selected_source": "cap10_three_fold_checkpoint_metric_replay_v2:wf_2020:bc_initialization",
    "source_selection": {
        "metric_replay": {
            "experiment": "cap10_three_fold_checkpoint_metric_replay_v2",
            "direct_comparison_baseline": "cap10_three_fold_checkpoint_metric_replay_v2",
            "source_card": contract["source_card"],
            "selected_daily_path": wf2020["selected_daily_path"],
        },
        "calibration": {
            "experiment": "wf2020_reference_floor_gate_calibration_v1",
            "source_card": contract["calibration_card"],
            "target_relative_return_by_fold_pp": calibrated_targets,
        },
    },
    "next_mechanism": NEXT_MECHANISM,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_training_allowed_after_target_revision": True,
}

wf2020_gap.to_csv(CARD / "wf2020_return_preserving_gap.csv", index=False)
(CARD / "target_revision_contract.json").write_text(json.dumps(json_safe(contract), indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "selection.json").write_text(json.dumps(json_safe(selection), indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(json_safe(summary), indent=2, sort_keys=True) + "\n", encoding="utf-8")

conclusion = f"""# CAP10 WF2020 Return-Preserving Target Revision V1

- Decision: `{DECISION}`.
- Promotion: blocked; this card retains no actor and starts no training.
- Source: `cap10_three_fold_checkpoint_metric_replay_v2` selected `wf_2020` `{wf2020['selected_candidate']}` with return `{float(source_candidate['test_return']):.2f}%`, relative `{float(wf2020['selected_relative_return']):.2f}pp`, and MaxDD `{float(wf2020['selected_max_dd']):.2f}%`.
- Root cause: `wf_2020` is under-participating, not over-drawing down. Drawdown headroom to the strict 20% gate is `{drawdown_headroom_to_20pct:.2f}pp`.
- Calibrated target: `wf_2020` relative floor `{float(calibrated_targets['wf_2020']):.2f}pp`; current replay gap `{relative_gap_to_calibrated_floor_pp:.2f}pp`.
- Baseline guardrail: pure `risk_cap_35_15_05` passes the calibrated `wf_2020` floor but has YTD return `{float(risk_cap_ytd['rulebase_return_pct']):.2f}%`, below the SOXLBacktest YTD reference `{float(gate_summary['reference_return_pct']):.2f}%` and current-best floor `{float(gate_summary['required_best_return_pct']):.2f}%`.
- Next action: `{NEXT_MECHANISM}`.

Sidecars: `wf2020_return_preserving_gap.csv`, `target_revision_contract.json`, `selection.json`, `summary.json`.
"""
(CARD / "conclusion.md").write_text(conclusion, encoding="utf-8")

summary


{'experiment': 'cap10_wf2020_return_preserving_target_revision_v1',
 'decision': 'wf2020_return_preserving_target_revision_ready_no_promotion',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'next_training_allowed_after_target_revision': True,
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
 'direct_comparison_baseline': 'cap10_three_fold_checkpoint_metric_replay_v2',
 'source_card': 'runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2/summary.json',
 'source_daily_path': 'runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2/official_daily_paths/wf_2020__bc_initialization_official_daily_path.csv',
 'failed_folds': ['wf_2020'],
 'primary_failure': 'wf_2020_return_underparticipation_after_metric_replay',
 'next_mechanism': 'build_return_preserving_wf2020_bull_participation_target_then_rerun_three_fold_bc_checkpoint_metric_replay',
 'target_max_dd_pct': 2

## Run Conclusion

This card is a target-revision handoff, not a promotion. The latest metric replay already keeps `wf_2020` MaxDD below the 20% strict gate, but it remains far below the calibrated return-participation floor. The next useful run should build a return-preserving `wf_2020` bull participation / re-entry target and then rerun the three-fold BC + checkpoint metric replay. Do not retain an actor until the retained model passes fixed-split, all required walk-forward windows, YTD return versus the SOXLBacktest reference, and the current-best return floor.
